# 08 · Buffalo model by region

07 showed that the model ranks places well **inside** regions with sightings (10 km block AUC 0.74), but hardly transfers to regions it never saw (quadrant AUC 0.62; camera agreement disappears when the camera region is held out). It also showed that about half the sightings sit in the southeast, around the Ngorongoro Crater.

This notebook follows that up in order:

1. **Where are the sightings?** Each point is assigned to its protected area (WDPA boundaries).
2. **Two landscapes:** the Serengeti ecosystem vs the crater highlands and rift. Does a model trained on one work on the other?
3. **Fairer regions and a less over-confident model:** balanced k-means regions instead of quadrants, and three XGBoost settings, on all data and on the Serengeti only.
4. **Camera traps** with their region held out, for every combination.

Self-contained: the data is built with `buffalo_data.py`, the metrics come from `buffalo_validation.py`.

## 0. Data

The same dataset as 06 and 07, built by `build_buffalo_dataset`. This notebook trains dozens of models, so it uses the download cache (`cache_dir="cache"`, git-ignored): the first run downloads, later runs reuse the files. Delete `cache/` to refresh.

Protected-area polygons come from the World Database on Protected Areas (WDPA) through Earth Engine, simplified to 100 m.

In [1]:
import ee
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

import buffalo_validation as bv
from buffalo_data import BAND_COLS, BuffaloConfig, build_buffalo_dataset

EE_PROJECT = "project-0cf410a3-f35f-4911-9c5"
ee.Initialize(project=EE_PROJECT)

config = BuffaloConfig(cache_dir="cache")
data = build_buffalo_dataset(config)

areas = bv.fetch_protected_areas()
points = bv.add_utm(data.sdm_df, config.cell_size_m)
points["area"] = bv.protected_area_of(points, areas)
cam = bv.add_utm(data.cameras, config.cell_size_m)
cam["area"] = bv.protected_area_of(cam, areas)

presences: loaded from cache\buffalo_presences_8be5ffc111.parquet
buffalo sightings in box: 674
presences after thinning: 658
mammals: loaded from cache\buffalo_mammals_8be5ffc111.parquet
mammal sightings in box: 15873
visited cells: 334


points_embedded: loaded from cache\buffalo_points_embedded_8be5ffc111.parquet
points missing embeddings: 0
cameras_embedded: loaded from cache\buffalo_cameras_embedded_8be5ffc111.parquet
dataset: 658 presences, 6580 background, 222 cameras


## 1. Where are the buffalo sightings?

In [2]:
by_area = pd.crosstab(points["area"], points["presence"]).rename(columns={0: "background", 1: "presences"})
by_area["share of presences"] = (by_area["presences"] / by_area["presences"].sum() * 100).round(1).astype(str) + " %"
print(by_area.sort_values("presences", ascending=False))
print()
print("camera traps:", cam["area"].value_counts().to_dict())

presence                 background  presences share of presences
area                                                             
Ngorongoro CA                  1323        311             47.3 %
Serengeti NP                   2433        232             35.3 %
outside protected areas        1607         56              8.5 %
Lake Manyara NP                  62         32              4.9 %
Mto wa Mbu GCA                   89         18              2.7 %
Grumeti GR                       88          3              0.5 %
Ikorongo GR                     125          3              0.5 %
Ikona WMA                       143          2              0.3 %
Loliondo GCA                    460          1              0.2 %
Maswa GR                         45          0              0.0 %
Lake Natron GCA                 169          0              0.0 %
Masai Mara NR                    36          0              0.0 %

camera traps: {'Serengeti NP': 222}


**Reading it:** the Ngorongoro Conservation Area holds the largest share of the sightings, more than Serengeti National Park. Most sightings outside protected areas also lie in the east, around Karatu and the rift. The crater floor, the highland forest and Lake Manyara are very different landscapes from the Serengeti plains. They're also major tourist stops, which is why they dominate GBIF. All camera traps sit inside Serengeti NP.

The study is therefore split into two landscapes, defined by protected areas only:
- **Serengeti ecosystem:** Serengeti NP and the reserves around it (Grumeti, Ikorongo, Ikona, Maswa, Loliondo, Masai Mara).
- **Crater highlands and rift:** Ngorongoro CA, Lake Manyara NP, Mto wa Mbu GCA.

Points outside protected areas are left out of both. Splitting them would need an arbitrary line.

**Background:** the original background was matched by year to *all* presences. Inside one landscape some years have fewer than 10 background points per presence. So each landscape keeps its presences and draws, per year, the same multiple of background (`ratio`, at most 10). The year distribution then still matches exactly.

In [3]:
from buffalo_validation import HIGHLAND_AREAS, SERENGETI_AREAS, year_matched

serengeti, serengeti_ratio = year_matched(points[points["area"].isin(SERENGETI_AREAS)])
highlands, highland_ratio = year_matched(points[points["area"].isin(HIGHLAND_AREAS)])
everything = points.reset_index(drop=True)

for name, df, ratio in [("Serengeti ecosystem", serengeti, serengeti_ratio), ("crater highlands + rift", highlands, highland_ratio),
                        ("all points (as in 07)", everything, 10)]:
    print(f"{name}: {int(df['presence'].sum())} presences, {int((df['presence'] == 0).sum())} background ({ratio} per presence)")

Serengeti ecosystem: 241 presences, 2410 background (10 per presence)
crater highlands + rift: 361 presences, 1083 background (3 per presence)
all points (as in 07): 658 presences, 6580 background (10 per presence)


## 2. Does habitat signal transfer between the two landscapes?

The most direct test of 07's finding: train on one landscape, test on the other.
- **Train Serengeti → test highlands**, and the reverse.
- For reference, each landscape is also scored on itself with 10 km block CV, the "seen region" number.

If the cross-landscape AUC is near 0.5 while the within-landscape AUC is clearly higher, buffalo sightings in the two landscapes look different in the embeddings. A single model is then mostly learning two separate local patterns, not one buffalo-habitat rule.

Default XGBoost settings here, as in 06.

In [4]:
def block_cv_auc(df, params=None):
    splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(df[BAND_COLS], df["presence"], groups=df["cell"]))
    table, _, _ = bv.blocked_cv(df, splits, [f"fold {i}" for i in range(5)], params)
    return table.loc["mean", "AUC"]


def transfer_auc(train, test, params=None):
    model = bv.make_model(train["presence"], params).fit(train[BAND_COLS], train["presence"])
    return roc_auc_score(test["presence"], model.predict_proba(test[BAND_COLS])[:, 1])


transfer = pd.DataFrame(
    {"within (10 km block CV)": [block_cv_auc(serengeti), block_cv_auc(highlands)],
     "transferred to the other landscape": [transfer_auc(serengeti, highlands), transfer_auc(highlands, serengeti)]},
    index=["trained on Serengeti", "trained on highlands"],
).round(3)
transfer

,within (10 km block CV),transferred to the other landscape
trained on Serengeti,0.687,0.438
trained on highlands,0.703,0.527


## 3. Fairer regions and a less over-confident model

**Regions:** 07's quadrants were lopsided (275 / 54 presences) because the sightings run diagonally. Here regions come from **k-means on the presence positions**: k centres placed where the sightings are, every point joins its nearest centre. Each region is held out once, so it's still a test on an unseen area, with more even folds.

**Model settings:** 07 found the default XGBoost over-confident (calibration slope 0.34; deep trees memorise local detail). Three settings are compared (`bv.XGB_SETTINGS`):
- **default:** depth 6, 100 trees, learning rate 0.3
- **regularized:** depth 3, 300 trees, learning rate 0.05, row and column subsampling, L2 penalty 5
- **strongly regularized:** depth 2, 400 trees, learning rate 0.03, at least 20 samples per leaf, L2 penalty 20

**Each combination of data and setting reports:**
- 10 km block AUC
- region AUC and TSS (cut-off chosen on the training regions)
- calibration slope on held-out region scores
- **camera rho:** each camera scored by the model whose held-out region contains it

The setting is chosen by **region AUC on the Serengeti data**. The camera rho is never used for choosing, so it stays an independent check.

In [5]:
DATASETS = {"all points": (everything, 5), "Serengeti only": (serengeti, 4)}

results, region_tables = {}, {}
for data_name, (df, k) in DATASETS.items():
    for setting, params in bv.XGB_SETTINGS.items():
        results[(data_name, setting)], region_tables[(data_name, setting)] = bv.region_evaluation(df, cam, k, params)

comparison = pd.DataFrame(results).T.drop(columns=["camera p"]).astype(float).round(3)
comparison.index.names = ["data", "setting"]
comparison

block AUC  region AUC  region TSS  \
data           setting                                                   
all points     default                   0.736       0.547       0.077   
               regularized               0.724       0.536       0.049   
               strongly regularized      0.710       0.517       0.009   
Serengeti only default                   0.687       0.576       0.085   
               regularized               0.715       0.589       0.113   
               strongly regularized      0.706       0.569       0.116   

                                     calibration slope  camera rho  \
data           setting                                               
all points     default                          -0.090      -0.141   
               regularized                      -0.228       0.033   
               strongly regularized             -0.295      -0.142   
Serengeti only default                           0.021       0.172   
               regularized                       0.086       0.300   
               strongly regularized             -0.057       0.033   

                                     median rate low  median rate medium  \
data           setting                                                     
all points     default                        10.838               2.600   
               regularized                     6.112               2.600   
               strongly regularized           10.428               2.005   
Serengeti only default                         2.510               2.773   
               regularized                     2.773               1.819   
               strongly regularized            3.667               5.520   

                                     median rate high  
data           setting                                 
all points     default                          5.420  
               regularized                      7.100  
               strongly regularized             5.300  
Serengeti only default                          7.487  
               regularized                     10.428  
               strongly regularized             4.800

In [6]:
best_setting = comparison.loc["Serengeti only", "region AUC"].idxmax()
print("chosen by region AUC on the Serengeti data:", best_setting)
print()
print("Serengeti only, regions held out one at a time:")
region_tables[("Serengeti only", best_setting)]

chosen by region AUC on the Serengeti data: regularized

Serengeti only, regions held out one at a time:


,presences,background,AUC,TSS,cut-off,Brier
held out,,,,,,
region 0,82.00,716.0,0.674,0.228,0.248,0.104
region 1,111.00,382.0,0.568,0.107,0.076,0.196
region 2,24.00,698.0,0.499,-0.068,0.178,0.165
region 3,24.00,614.0,0.617,0.184,0.232,0.196
mean,60.25,602.5,0.589,0.113,0.183,0.165


**How to read the comparison:**
- **Block AUC vs region AUC:** the gap is how much the model relies on having seen the area. A setting that narrows the gap transfers better.
- **Calibration slope:** closer to 1 is better. Regularization should raise it from the default.
- **Camera rho:** the independent check. Positive with the camera region held out means the model ranks camera sites correctly in an area it never trained on.
- **All points vs Serengeti only:** if the Serengeti-only model transfers between regions and agrees with the cameras better, mixing in the crater highlands was hurting the Serengeti model.